# Create bronze tables 
1. Use this notebook to create bronze lake tables. 
2. Select **Run all** to run the notebook. 
3. This will overwrite the data in the bronze layer 
4. When the notebook run is completed, return to your lakehouse and refresh your lake views graph. 


In [ ]:
# ── Parameters ─────────────────────────────────────────────────
ROOT_PATH = "Files/wmpp-production-data-export-birmingham/latest"       # Shortcut path in lakehouse. Path to the "latest" folder inside your shortcut
BRONZE_SCHEMA    = "bronze"             # Schema for bronze tables
TABLE_PREFIX     = "brz_"               # Prefix for bronze tables
LOAD_MODE        = "append"             # append | overwrite
TEXT_QUALIFIER   = '"'                  # CSV text qualifier character
REBUILD   = 0                           # Rebuild Bronze tables

print(f"ROOT_PATH=[{ROOT_PATH}]")
print(f"BRONZE_SCHEMA=[{BRONZE_SCHEMA}]")
print(f"TABLE_PREFIX=[{TABLE_PREFIX}]")
print(f"LOAD_MODE=[{LOAD_MODE}]")
print(f"TEXT_QUALIFIER=[{TEXT_QUALIFIER}]")
print(f"REBUILD=[{REBUILD}]")


StatementMeta(, 9aa61b4d-b79b-4f19-8d87-0dfbdca464f1, 3, Finished, Available, Finished, False)

ROOT_PATH=[Files/wmpp-production-data-export-birmingham/latest]
BRONZE_SCHEMA=[bronze]
TABLE_PREFIX=[brz_]
LOAD_MODE=[append]
TEXT_QUALIFIER=["]
REBUILD=[0]


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import uuid
from notebookutils import mssparkutils
import os

table_count = 0
BATCH_ID = str(uuid.uuid4())

try:
    folders = mssparkutils.fs.ls(ROOT_PATH)

    for f in folders:
        # inside loop after successful write
        table_count += 1
        folder_name = os.path.basename(f.path.rstrip("/"))

        clean_name = folder_name.split(".")[-2]
        clean_name = f"{BRONZE_SCHEMA}.{clean_name}"

        table_path = f"{ROOT_PATH}/{folder_name}"
        
        print(f"Processing folder: {folder_name} -> table: {clean_name}")
        if REBUILD==1:
            print(f"\tDropping table: {clean_name}")
            sql_drop= f"DROP TABLE IF EXISTS {clean_name}"
            spark.sql(sql_drop)

        # Try parquet first (most common)
        if folder_name.lower().endswith(".parquet"):
            df = spark.read.format("parquet").load(table_path)
            print(f"\tLoaded parquet for {folder_name}")
        elif folder_name.lower().endswith(".csv"):
            # Try CSV if parquet fails
            try:
                df = (
                        spark.read
                        .format("csv")
                        .option("header", "true")
                        .option("quote", TEXT_QUALIFIER)
                        .option("escape", TEXT_QUALIFIER)
                        .option("multiLine", "true")
                        .load(table_path)
                    )
                print(f"\tLoaded CSV for {folder_name}")
            except Exception as e:
                print(f"FAILED: {folder_name}")
                print(type(e).__name__)
                print(str(e))
                #print(f"Skipping {folder_name}, unsupported format or empty folder")
                continue
                
        # Keep Bronze string-oriented while carrying lineage and the source export timestamp.
        df = (df.withColumn("_ingestion_timestamp", F.current_timestamp())
                .withColumn("_source_file", F.lit(folder_name))
                .withColumn("_ingestion_id", F.lit(BATCH_ID)))
        if "export_date" not in df.columns:
            df = df.withColumn("export_date", F.current_timestamp().cast("string"))
        else:
            df = df.withColumn("export_date", F.coalesce(F.col("export_date").cast("string"), F.current_timestamp().cast("string")))

        # Write to Lakehouse as Delta table
        try:
            df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(clean_name)
            print(f"\tCreated/updated table: {clean_name}")
        except Exception as e:
            print(f"\tWrite failed for {clean_name}")
            print(str(e))
            raise
            
except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR:", str(e))
    raise

print(f"Successfully processed {table_count} tables")   
print("All tables processed successfully!")


StatementMeta(, 9aa61b4d-b79b-4f19-8d87-0dfbdca464f1, 4, Finished, Available, Finished, False)

Processing folder: additional_fee.csv -> table: bronze.additional_fee
	Loaded CSV for additional_fee.csv


	Created/updated table: bronze.additional_fee
Processing folder: foster_carer.csv -> table: bronze.foster_carer
	Loaded CSV for foster_carer.csv


	Created/updated table: bronze.foster_carer
Processing folder: foster_home.csv -> table: bronze.foster_home
	Loaded CSV for foster_home.csv


	Created/updated table: bronze.foster_home
Processing folder: foster_transport.csv -> table: bronze.foster_transport
	Loaded CSV for foster_transport.csv


...

	Created/updated table: bronze.referral_provider_message
Processing folder: referral_provider.csv -> table: bronze.referral_provider
	Loaded CSV for referral_provider.csv


	Created/updated table: bronze.referral_provider
Processing folder: referral_spot_category.csv -> table: bronze.referral_spot_category
	Loaded CSV for referral_spot_category.csv


	Created/updated table: bronze.referral_spot_category
Processing folder: referral.csv -> table: bronze.referral
	Loaded CSV for referral.csv


	Created/updated table: bronze.referral
Processing folder: supervising_social_worker.csv -> table: bronze.supervising_social_worker
	Loaded CSV for supervising_social_worker.csv


	Created/updated table: bronze.supervising_social_worker
Successfully processed 33 tables
All tables processed successfully!


In [ ]:
# Add missing table that do not appear in the latest batch

framework_path = "Files/deprecated_wmpp_files/framework.csv"
# extract framework
df = (
                        spark.read
                        .format("csv")
                        .option("header", "true")
                        .option("quote", TEXT_QUALIFIER)
                        .option("escape", TEXT_QUALIFIER)
                        .option("multiLine", "true")
                        .load(framework_path)
                    )

clean_name = f"{BRONZE_SCHEMA}.framework"
df = (df.withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file", F.lit("framework.csv"))
        .withColumn("_ingestion_id", F.lit(BATCH_ID))
        .withColumn("export_date", F.current_timestamp().cast("string")))


try:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(clean_name)
    print(f"\tCreated/updated table: {clean_name}")
except Exception as e:
    print(f"\tWrite failed for {clean_name}")
    print(str(e))
    raise

    

StatementMeta(, 9aa61b4d-b79b-4f19-8d87-0dfbdca464f1, 5, Finished, Available, Finished, False)

	Created/updated table: bronze.framework
